# Media Changes
See: https://phabricator.wikimedia.org/T299712

## Imports / settings / etc.

In [ ]:
import re

import wmfdata

In [ ]:
spark = wmfdata.spark.get_session(app_name='pyspark regular; media-changes',
                                  type='yarn-large', # local, yarn-regular, yarn-large
                                  )  

In [ ]:
print("Mediawiki partitions:")
spark.sql("show partitions wmf_raw.mediawiki_project_namespace_map").show(50, False)

## Parameters

In [ ]:
mw_snapshot = '2021-12'  # data will include this month -- e.g., 2020-07 means data is up to 31 July 2020 (at least)
wiki_dbs = ('ruwiki', )
start_timestamp = '2021-12'  # example format: 2008-04-04 15:04:29.0
media_list_table = 'isaacj.media_ruwiki_dec_2021'

## Utils

In [ ]:
MEDIA_PREFIXES = ['File', 'Image', 'Media']

MEDIA_ALIASES = {
    "ab": ["?????", "????", "?????", "??????", "???????????"],
    "ace": ["Beureukaih", "Gambar", "Alat", "Berkas"],
    "ady": ["?????"],
    "af": ["L?er", "Beeld"],
    "als": ["Medium", "Datei", "Bild"],
    "am": ["???", "???"],
    "an": ["Imachen", "Imagen"],
    "ang": ["Ymele", "Bili?"],
    "ar": ["?????", "????", "?????", "???"],
    "arc": ["????", "?????"],
    "arz": ["?????", "????", "?????", "???"],
    "as": ["?????", "?????", "?????", "??????"],
    "ast": ["Imaxen", "Ficheru", "Imaxe", "Archivu", "Imagen", "Medios"],
    "atj": ["Tipatcimoctakewin", "Natisinahikaniwoc"],
    "av": ["?????", "????", "???????????"],
    "ay": ["Medio", "Archivo", "Imagen"],
    "az": ["Mediya", "??kil", "Fayl"],
    "azb": ["?????", "?????", "????", "????", "????????"],
    "ba": ["?????", "?????", "????", "???????????"],
    "bar": ["Medium", "Datei", "Bild"],
    "bat-smg": ["Vaizdas", "Med?j?", "Abruozdielis"],
    "bcl": ["Medio", "Ladawan"],
    "be": ["???????????", "????", "?????"],
    "be-x-old": ["?????", "????", "?????"],
    "bg": ["?????", "????", "????????"],
    "bh": ["??????", "?????"],
    "bjn": ["Barakas", "Gambar", "Berkas"],
    "bm": ["M?dia", "Fichier"],
    "bn": ["?????", "???????"],
    "bpy": ["???", "???????"],
    "br": ["Skeudenn", "Restr"],
    "bs": ["Mediji", "Slika", "Datoteka", "Medija"],
    "bug": ["Gambar", "Berkas"],
    "bxr": ["????", "????", "???????????"],
    "ca": ["Fitxer", "Imatge"],
    "cbk-zam": ["Medio", "Archivo", "Imagen"],
    "cdo": ["??", "??", "??", "??"],
    "ce": ["?l??", "?????", "????", "????", "?????", "???????????"],
    "ceb": ["Payl", "Medya", "Imahen"],
    "ch": ["Litratu"],
    "ckb": ["?????", "?????"],
    "co": ["Immagine"],
    "crh": ["?????", "Resim", "????", "Fayl", "?????"],
    "cs": ["Soubor", "M?dia", "Obr?zok"],
    "csb": ["?br?zk", "Grafika"],
    "cu": ["????", "?????", "????", "?????????"],
    "cv": ["?????", "???????", "???????????"],
    "cy": ["Delwedd"],
    "da": ["Billede", "Fil"],
    "de": ["Medium", "Datei", "Bild"],
    "din": ["Ci?l", "Apamdu??t"],
    "diq": ["Medya", "Dosya"],
    "dsb": ["Wobraz", "Dataja", "Bild", "Medija"],
    "dty": ["?????", "??????"],
    "dv": ["??????", "??????", "??????"],
    "el": ["??????", "??????", "????", "?????"],
    "eml": ["Immagine"],
    "eo": ["Dosiero", "A?dvida?o"],
    "es": ["Medio", "Archivo", "Imagen"],
    "et": ["Pilt", "Fail", "Meedia"],
    "eu": ["Irudi", "Fitxategi"],
    "ext": ["Archivu", "Imagen", "Mediu"],
    "fa": ["?????", "?????", "????", "??????", "????????"],
    "ff": ["M?dia", "Fichier"],
    "fi": ["Kuva", "Tiedosto"],
    "fiu-vro": ["Pilt", "Meedi?"],
    "fo": ["Mi?il", "Mynd"],
    "fr": ["M?dia", "Fichier"],
    "frp": ["?m?ge", "Fichi?r", "M?dia"],
    "frr": ["Medium", "Datei", "Bild"],
    "fur": ["Immagine", "Figure"],
    "fy": ["Ofbyld"],
    "ga": ["?omh?", "Me?n"],
    "gag": ["Mediya", "Medya", "Resim", "Dosya", "Dosye"],
    "gan": ["????", "??", "??", "??", "??", "??", "??", "??", "??"],
    "gd": ["Faidhle", "Meadhan"],
    "gl": ["Imaxe", "Ficheiro", "Arquivo", "Imagem"],
    "glk": ["?????", "?????", "??????", "????", "????????", "????"],
    "gn": ["Medio", "Imagen", "Ta'?nga"],
    "gom": ["??????", "??????", "????"],
    "gor": ["Gambar", "Berkas"],
    "got": ["??????????"],
    "gu": ["??????-??????? (??????)", "??????-???????_(??????)", "?????"],
    "gv": ["Coadan", "Meanyn"],
    "hak": ["??", "??", "??", "??"],
    "haw": ["Ki?i", "Waihona", "P?paho"],
    "he": ["?????", "??", "????", "????"],
    "hi": ["??????", "?????"],
    "hif": ["file", "saadhan"],
    "hr": ["Mediji", "DT", "Slika", "F", "Datoteka"],
    "hsb": ["Wobraz", "Dataja", "Bild"],
    "ht": ["Imaj", "Fichye", "Medya"],
    "hu": ["K?p", "F?jl", "M?dia"],
    "hy": ["??????", "?????"],
    "ia": ["Imagine", "Multimedia"],
    "id": ["Gambar", "Berkas"],
    "ig": ["Nk?", "Midia", "Us?r?", "?kw?kw? or?notu", "?kw?kw?_or?notu"],
    "ii": ["????", "??", "??", "??", "??"],
    "ilo": ["Midia", "Papeles"],
    "inh": ["?????", "????", "???????????"],
    "io": ["Imajo", "Arkivo"],
    "is": ["Mi?ill", "Mynd"],
    "it": ["Immagine"],
    "ja": ["????", "????", "??"],
    "jbo": ["velsku", "datnyvei"],
    "jv": ["Barkas", "Medhia", "Gambar", "M?dhia"],
    "ka": ["?????", "??????", "?????"],
    "kaa": ["Swret", "?????", "?????", "Taspa", "Su'wret", "?????", "?????"],
    "kab": ["Tugna"],
    "kbd": ["?????", "????"],
    "kbp": ["M?dia", "Fichier"],
    "kg": ["Fisye"],
    "kk": ["Swret", "?????", "?????", "Taspa", "?????", "?????"],
    "kl": ["Billede", "Fiileq", "Fil"],
    "km": ["?????", "??????", "????", "????"],
    "kn": ["?????", "?????"],
    "ko": ["???", "??", "??"],
    "koi": ["?????", "????", "???????????"],
    "krc": ["?????", "????", "???????????"],
    "ks": ["?????", "?????"],
    "ksh": ["Beld", "Meedije", "Medie", "Belld", "Medium", "Datei", "Meedijum", "Bild"],
    "ku": ["?????", "?????", "Medya", "W?ne"],
    "kv": ["?????", "????", "???????????"],
    "kw": ["Restren"],
    "ky": ["?????", "????"],
    "la": ["Imago", "Fasciculus"],
    "lad": ["Dossia", "Medya", "Archivo", "Dosya", "Imagen", "Meddia"],
    "lb": ["Fichier", "Bild"],
    "lbe": ["?????", "?????", "???????????"],
    "lez": ["?????", "Mediya", "????", "??kil", "???????????"],
    "lfn": ["Fix"],
    "li": ["Afbeelding", "Plaetje", "Aafbeilding"],
    "lij": ["Immaggine", "Immagine"],
    "lmo": ["Immagine", "Im?jine", "Archivi"],
    "ln": ["M?dia", "Fichier"],
    "lo": ["????", "???", "???"],
    "lrc": ["?????", "?????", "????????", "?????", "???", "????????"],
    "lt": ["Vaizdas", "Medija"],
    "ltg": ["Medeja", "Fails"],
    "lv": ["Att?ls"],
    "mai": ["??????", "????"],
    "map-bms": ["Barkas", "Medhia", "Gambar", "M?dhia"],
    "mdf": ["?????", "????", "???????????"],
    "mg": ["Rakitra", "Sary", "M?dia"],
    "mhr": ["?????", "????", "???????????"],
    "min": ["Gambar", "Berkas"],
    "mk": ["??????????", "??????", "??????", "?????"],
    "ml": ["???????", "??", "?????", "???", "??????"],
    "mn": ["?????", "????", "?????"],
    "mr": ["?????", "??????"],
    "mrj": ["?????", "????", "???????????"],
    "ms": ["Fail", "Imej"],
    "mt": ["Midja", "Medja", "Stampa"],
    "mwl": ["Multim?dia", "Fexeiro", "Ficheiro", "Arquivo", "Imagem"],
    "my": ["?????", "??????"],
    "myv": ["?????", "???????", "???????????"],
    "mzn": ["?????", "?????", "??????", "????", "??????", "????????"],
    "nah": ["M?diatl", "?xiptli", "Imagen"],
    "nap": ["Fi?ra", "Immagine"],
    "nds": ["Datei", "Bild"],
    "nds-nl": ["Ofbeelding", "Afbeelding", "Bestaand"],
    "ne": ["??????", "?????"],
    "new": ["????", "??????"],
    "nl": ["Bestand", "Afbeelding"],
    "nn": ["Fil", "Bilde", "Filpeikar"],
    "no": ["Fil", "Medium", "Bilde"],
    "nov": [],
    "nrm": ["M?dia", "Fichier"],
    "nso": ["Seswant?ho"],
    "nv": ["E?elyaa?g??"],
    "oc": ["Imatge", "Fichi?r", "M?dia"],
    "olo": ["Kuva", "Medii", "Failu"],
    "or": ["??????", "????"],
    "os": ["???", "?????", "????", "???????????"],
    "pa": ["?????", "?????"],
    "pcd": ["M?dia", "Fichier"],
    "pdc": ["Medium", "Datei", "Bild", "Feil"],
    "pfl": ["Dadai", "Medium", "Datei", "Bild"],
    "pi": ["??????", "?????"],
    "pl": ["Plik", "Grafika"],
    "pms": ["Figura", "Immagine"],
    "pnb": ["?????", "?????", "????"],
    "pnt": ["??????", "???????", "???????", "?????"],
    "ps": ["?????", "????", "?????"],
    "pt": ["Multim?dia", "Ficheiro", "Arquivo", "Imagem"],
    "qu": ["Midya", "Imagen", "Rikcha"],
    "rm": ["Multimedia", "Datoteca"],
    "rmy": ["Fi?ier", "Mediya", "Chitro", "Imagine"],
    "ro": ["Fi?ier", "Imagine", "Fi?ier"],
    "roa-rup": ["Fi?ier", "Imagine", "Fi?ier"],
    "roa-tara": ["Immagine"],
    "ru": ["?????", "????", "???????????"],
    "rue": ["?????", "?????", "????", "???????????", "??????????"],
    "rw": ["Dosiye", "Itangazamakuru"],
    "sa": ["???????", "????????", "???????", "??????", "??????"],
    "sah": ["????????", "????", "????", "???????????"],
    "sat": ["???", "??????"],
    "sc": ["Imm?gini"],
    "scn": ["Immagine", "Mm?ggini", "M?dia"],
    "sd": ["???", "??????", "????"],
    "se": ["Fiila"],
    "sg": ["M?dia", "Fichier"],
    "sh": ["Mediji", "Slika", "??????", "Datoteka", "Medija", "?????"],
    "si": ["????", "???????", "?????"],
    "sk": ["S?bor", "Obr?zok", "M?di?"],
    "sl": ["Slika", "Datoteka"],
    "sq": ["Figura", "Skeda"],
    "sr": ["????????", "Medij", "Slika", "??????", "Datoteka", "?????", "Medija", "?????"],
    "srn": ["Afbeelding", "Gefre"],
    "stq": ["Bielde", "Bild"],
    "su": ["M?dia", "Gambar"],
    "sv": ["Fil", "Bild"],
    "sw": ["Faili", "Picha"],
    "szl": ["Plik", "Grafika"],
    "ta": ["??????", "?????"],
    "tcy": ["???????", "????"],
    "te": ["????", "???????", "?????", "??????"],
    "tet": ["Imajen", "Arquivo", "Imagem"],
    "tg": ["???", "?????"],
    "th": ["????", "????", "???"],
    "ti": ["???", "???"],
    "tk": ["Fa?l"],
    "tl": ["Midya", "Talaksan"],
    "tpi": ["Fail"],
    "tr": ["Medya", "Resim", "Dosya", "Ortam"],
    "tt": ["?????", "?????", "????", "R?sem", "???????????"],
    "ty": ["M?dia", "Fichier"],
    "tyv": ["?????", "????", "???????????"],
    "udm": ["?????", "????", "?????", "???????????"],
    "ug": ["??????", "??????"],
    "uk": ["?????", "?????", "????", "???????????", "??????????"],
    "ur": ["?????", "?????", "????", "?????", "????", "???"],
    "uz": ["Mediya", "Tasvir", "Fayl"],
    "vec": ["Immagine", "Im?jine", "M?dia"],
    "vep": ["Pilt", "Fail"],
    "vi": ["Ph??ng_ti?n", "T?p_tin", "H?nh", "T?p tin", "Ph??ng ti?n"],
    "vls": ["Afbeelding", "Ofbeeldienge"],
    "vo": ["Ragiv", "Magod", "N?namakan?d"],
    "wa": ["Im?dje"],
    "war": ["Medya", "Fayl", "Paypay"],
    "wo": ["Xibaarukaay", "Dencukaay"],
    "wuu": ["??", "??", "??", "??"],
    "xal": ["????", "?????", "???????????", "????"],
    "xmf": ["?????", "??????", "?????"],
    "yi": ["?????", "?????", "????", "????"],
    "yo": ["F??l?", "Am?h?nm?w?r?n", "?w?r?n"],
    "za": ["????", "??", "??", "??", "??"],
    "zea": ["Afbeelding", "Plaetje"],
    "zh": ["????", "F", "??", "??", "??", "??", "??", "??", "??"],
    "zh-classical": ["??", "??", "??", "??"],
    "zh-min-nan": ["t?ng-?n", "??", "??", "M?i-th?", "??", "??"],
    "zh-yue": ["?", "?", "??", "?", "??", "?", "??", "??", "??", "??", "??"],
}

# https://commons.wikimedia.org/wiki/Commons:File_types
IMAGE_EXTENSIONS = ['.jpg', '.png', '.svg', '.gif', '.jpeg', '.tif', '.bmp', '.webp', '.xcf']
VIDEO_EXTENSIONS = ['.ogv', '.webm', '.mpg', '.mpeg']
AUDIO_EXTENSIONS = ['.ogg', '.mp3', '.mid', '.webm', '.flac', '.wav', '.oga']
MEDIA_EXTENSIONS = list(set(IMAGE_EXTENSIONS + VIDEO_EXTENSIONS + AUDIO_EXTENSIONS))

In [ ]:
exten_regex = ('(' + '|'.join([e + r'\b' for e in MEDIA_EXTENSIONS]) + ')').replace('.', r'\.')
extension_pattern = re.compile(fr'([\w ,().-]+){exten_regex}', flags=re.UNICODE)
bracket_pattern = re.compile(r'(?<=\[\[)(.*?)(?=]])', flags=re.DOTALL)

# NOTE: I explored several approaches to this function and how they impacted speed:
# * mwparserfromhell parsing substantially increases processing time, even compared to many regexes
# * Reducing down the number of extensions considered has a very minimal impact on time
# * Removing the first regex that extracts links has a very minimal impact on time. In theory it should be mostly unnecessary but will catch some rare file extensions.
# * Ignoring upper-case file extensions (e.g., .JPG) by not lower-casing the wikitext and just doing .findall over the iterative .search has very little impact on time

def getMedia(wikitext, wiki_db='enwiki', max_link_length=240):
    """Gather counts of media files found directly in wikitext.
    
    See https://phabricator.wikimedia.org/T299712 for more details.
    Link length: https://commons.wikimedia.org/wiki/Commons:File_naming#Length
    """
    lang = wiki_db.replace('wiki', '')
    try:
        # find standard bracket-syntax links -- this likely could be dropped but adds minimal overhead
        med_prefixes = MEDIA_PREFIXES + MEDIA_ALIASES.get(lang, [])
        links = bracket_pattern.findall(wikitext)
        bracket_links = set([l.split(':', maxsplit=1)[1].split('|', maxsplit=1)[0].strip() for l in links if l.split(':', maxsplit=1)[0] in med_prefixes])
        
        # supplement with links outside brackets as determined via known file extensions
        # lower-case to handle e.g., .JPG instead of .jpg when searching for file extensions
        lc_wt = wikitext.lower()
        exten_links = []
        end = 0
        while True:
            m = extension_pattern.search(lc_wt, pos=end)
            if m is None:
                break
            start, end = m.span()
            exten_links.append(wikitext[start:end].strip())
        return [l.replace('\n', ' ') for l in bracket_links.union(exten_links) if len(l) <= max_link_length]
    except Exception:
        return None
    
spark.udf.register('getMedia', getMedia, 'ARRAY<String>')

In [ ]:
def compareMediaLists(curr_media, prev_media):
    """Compare two media lists to determine what changed."""
    try:
        changes = []
        unaligned = set(curr_media) ^ set(prev_media)
        for m in unaligned:
            if m in curr_media:
                changes.append((m, 1))
            elif m in prev_media:
                changes.append((m, -1))
        return changes
    except Exception:
        return None
    
spark.udf.register('compareMediaLists', compareMediaLists, 'ARRAY<STRUCT<filename:STRING, action:INT>>')

In [ ]:
# Future TODO: partition by wiki_db? If so, also run:
# spark.sql("SET hive.exec.dynamic.partition.mode = nonstrict")

create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {media_list_table} (
        wiki_db                         STRING         COMMENT 'Wiki -- e.g., enwiki for English',
        page_id                         INT            COMMENT 'Article page ID',
        user_text                       STRING         COMMENT 'User name of who made edit',
        revision_id                     BIGINT         COMMENT 'Revision ID',
        revision_parent_id              BIGINT         COMMENT 'Revision ID of parent revision',
        revision_is_identity_reverted   BOOLEAN        COMMENT 'Was revision reverted?',
        revision_is_identity_revert     BOOLEAN        COMMENT 'Did revision restore a previous revision?',
        revision_tags                   ARRAY<STRING>  COMMENT 'Edit tags associated with revision',
        cur_rev_media_array             ARRAY<STRING>  COMMENT 'List of images in current revision',
        par_rev_media_array             ARRAY<STRING>  COMMENT 'List of images in parent revision'
    )
    """

spark.sql(create_table_query)
print(create_table_query)

## Generate media lists for December 2021 Russian Wikipedia edits

In [ ]:
"""
Explanation of CTEs:
* revisions: get all revisions + metadata from desired wikis / timeframe.
  * only main articles and filter out bots / anonymous users
* all_revision_ids: build deduplicated lists of all revision + parent revision IDs
* media_lists: for each revision ID, extract images from associated wikitext
* INSERT OVERWRITE...: join back in media lists with revisions + metadata

# TODO: are newlines in revision_text causing NULLs?
"""

print_for_hive = False
do_execute = True
wiki_db_str = "('" + "','".join(wiki_dbs) + "')"  # otherwise single wiki leads to confusing syntax errors

query = f"""
WITH revisions AS (
    SELECT
      wiki_db,
      page_id,
      event_user_text AS user_text,
      revision_id,
      revision_parent_id,
      revision_is_identity_reverted,
      revision_is_identity_revert,
      revision_tags
    FROM wmf.mediawiki_history
    WHERE
      snapshot = '{mw_snapshot}'
      AND wiki_db IN {wiki_db_str}
      AND page_namespace = 0
      AND event_type = 'create'
      AND event_entity = 'revision'
      AND event_timestamp >= '{start_timestamp}'
      AND SIZE(event_user_is_bot_by) < 1
      AND NOT event_user_is_anonymous
      AND NOT page_is_redirect
),
all_revision_ids AS (
    SELECT DISTINCT
      wiki_db,
      rev_id
    FROM (
        SELECT
          wiki_db,
          revision_id AS rev_id
        FROM revisions
        UNION ALL
        SELECT
          wiki_db,
          revision_parent_id AS rev_id
        FROM revisions
    ) r
),
media_lists AS (
    SELECT
      r.wiki_db,
      r.rev_id,
      getMedia(revision_text, wt.wiki_db) AS media_array
    FROM wmf.mediawiki_wikitext_history wt
    INNER JOIN all_revision_ids r
      ON (wt.wiki_db = r.wiki_db
          AND wt.revision_id = r.rev_id)
    WHERE
      snapshot = '{mw_snapshot}'
      AND wt.wiki_db IN {wiki_db_str}
)
INSERT OVERWRITE TABLE {media_list_table}     
SELECT
  r.wiki_db,
  page_id,
  user_text,
  revision_id,
  revision_parent_id,
  revision_is_identity_reverted,
  revision_is_identity_revert,
  revision_tags,
  c.media_array AS cur_rev_media_array,
  p.media_array AS par_rev_media_array
FROM revisions r
LEFT JOIN media_lists c
  ON (r.wiki_db = c.wiki_db
      AND r.revision_id = c.rev_id)
LEFT JOIN media_lists p
  ON (r.wiki_db = p.wiki_db
      AND r.revision_parent_id = p.rev_id)
"""

if print_for_hive:
    print(re.sub(' +', ' ', re.sub('\n', ' ', query)).strip())
else:
    print(query)

if do_execute:
    result = spark.sql(query)

In [ ]:
# Example data: for action, 1 = added; -1 = removed
spark.sql(f'SELECT wiki_db, page_id, user_text, revision_id, revision_is_identity_reverted AS was_reverted, revision_is_identity_revert AS is_revert, revision_tags, INLINE(compareMediaLists(cur_rev_media_array, par_rev_media_array)) FROM {media_list_table} LIMIT 25').show(50, False)


## Evaluation

In [ ]:
# Descriptive stats on total edits etc. in dataset

print_for_hive = False
do_execute = True

query = f"""
    SELECT
      wiki_db,
      SUM(IF(revision_is_identity_reverted, 1, 0)) AS total_num_reverted,
      COUNT(1) AS total_num_edits,
      COUNT(DISTINCT(user_text)) AS total_num_editors
    FROM {media_list_table}
     WHERE
      revision_id IS NOT NULL
    GROUP BY
      wiki_db
"""

if print_for_hive:
    print(re.sub(' +', ' ', re.sub('\n', ' ', query)).strip())
else:
    print(query)

if do_execute:
    spark.sql(query).show(500, False)

In [ ]:
# Descriptive stats on subset of edits that are image changes

print_for_hive = False
do_execute = True

query = f"""
WITH num_edits AS (
    SELECT
      wiki_db,
      SUM(IF(revision_is_identity_reverted, 1, 0)) AS total_num_reverted,
      COUNT(1) AS total_num_edits,
      COUNT(DISTINCT(user_text)) AS total_num_editors
    FROM {media_list_table}
     WHERE
      revision_id IS NOT NULL
    GROUP BY
      wiki_db
),
changes AS (
    SELECT
      wiki_db,
      user_text,
      revision_id,
      revision_is_identity_reverted AS was_reverted,
      INLINE(compareMediaLists(cur_rev_media_array, par_rev_media_array))
    FROM {media_list_table}
    WHERE
      revision_id IS NOT NULL
),
change_counts AS (
    SELECT
      wiki_db,
      was_reverted,
      COUNT(DISTINCT(revision_id)) AS num_edits,
      COUNT(DISTINCT(user_text)) AS num_editors,
      SUM(IF(action = 1, 1, 0)) AS images_added,
      SUM(IF(action = -1, 1, 0)) AS images_removed
    FROM changes
    GROUP BY
      wiki_db,
      was_reverted
)
SELECT
  c.wiki_db,
  was_reverted,
  num_edits,
  ROUND(100 * num_edits / total_num_edits, 3) AS pct_edits,
  num_editors,
  ROUND(100 * num_editors / total_num_editors, 3) AS pct_editors,
  images_added,
  images_removed
FROM change_counts c
INNER JOIN num_edits t
  ON (c.wiki_db = t.wiki_db)

"""

if print_for_hive:
    print(re.sub(' +', ' ', re.sub('\n', ' ', query)).strip())
else:
    print(query)

if do_execute:
    spark.sql(query).show(500, False)